# Fase 6: comparación de las tres variantes

Mide `build/hough` (global), `build/hough_const` (global + constante) y `build/hough_final` (global + constante + compartida) en la **misma sesión y GPU**, con CUDA Events alrededor de cada lanzamiento y validación contra la CPU en cada ejecución. Usa el mismo protocolo de la fase 4:

1. se estabiliza la GPU con 2000 ejecuciones descartadas;
2. cada ejecución hace 1 calentamiento + **100 lanzamientos medidos** y se reportan los **últimos 10**;
3. se hacen **6 rondas**, una por cada permutación del orden de las tres variantes. Así cada variante queda dos veces en cada posición (primera, segunda y tercera) y se cancelan la deriva de la GPU y el efecto de la variante anterior;
4. se registra el estado de la GPU antes y después de cada ejecución.

Resultado: 60 mediciones reportadas por variante. Usa **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.

Antes de ejecutar, `src/houghSharedConst.cu` y el objetivo `build/hough_final` del Makefile deben estar en la rama indicada en `RAMA`.

In [ ]:
!nvidia-smi
!nvcc --version

## Proyecto

In [ ]:
from pathlib import Path
import subprocess, os
RAMA = 'main'
root = Path('/content/P3-CUDA-Paralela')
if not root.exists():
    subprocess.run(['git', 'clone', '--branch', RAMA,
                    'https://github.com/Jorge162017/P3-CUDA-Paralela.git', str(root)], check=True)
os.chdir(root)
subprocess.run(['git', 'fetch', 'origin', RAMA], check=True)
subprocess.run(['git', 'checkout', RAMA], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', RAMA], check=True)
subprocess.run(['make', 'build/hough', 'build/hough_const', 'build/hough_final'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Rama:', RAMA, '| Commit:', commit)

## Seis rondas, cien lanzamientos por ejecución; se reportan los últimos diez

Cada ejecución escribe su CSV completo (100 filas) en `build/`. A `results/fase6-tres-memorias/final/` se copian solo las 10 mediciones reportadas (91 a 100), con las mismas columnas que las bitácoras anteriores. El log resume cada ejecución con el promedio de los lanzamientos 1–10, 11–90 y 91–100 y se guarda aunque falle una ejecución.

In [ ]:
import shutil
import pandas as pd
from itertools import permutations
out = Path('results/fase6-tres-memorias/final')
LANZAMIENTOS, REPORTADOS = 100, 10
shutil.rmtree(out, ignore_errors=True)
out.mkdir(parents=True)
PROGRAMA = {'global': './build/hough', 'constante': './build/hough_const',
            'tres_memorias': './build/hough_final'}
ARCHIVO = {'global': 'global-times.csv', 'constante': 'constante-times.csv',
           'tres_memorias': 'tres-memorias-times.csv'}
NOMBRES = list(PROGRAMA)
ORDEN = {f'ronda{i}': list(p) for i, p in enumerate(permutations(NOMBRES), 1)}
CONSULTA = ['nvidia-smi', '--query-gpu=pstate,clocks.sm,clocks.mem,temperature.gpu,power.draw',
            '--format=csv,noheader,nounits']
log = subprocess.check_output(['nvidia-smi'], text=True)
log += subprocess.check_output(['nvcc', '--version'], text=True)
log += f'Rama: {RAMA}\nCommit: {commit}\n'
log += 'Compilacion: make build/hough build/hough_const build/hough_final; -O2 -std=c++17 --fmad=false\n'
log += 'Protocolo: 2000 ejecuciones descartadas para estabilizar la GPU; luego 6 rondas (todas las permutaciones de las 3 variantes), '
log += f'cada ejecucion con 1 calentamiento + {LANZAMIENTOS} mediciones; se reportan las ultimas {REPORTADOS}\n'
estabilizar = subprocess.run([PROGRAMA['global'], 'data/runway.pgm', '--runs', '2000'],
                             text=True, capture_output=True)
log += f'== estabilizacion (descartada): codigo {estabilizar.returncode}; '
log += estabilizar.stdout.splitlines()[-2] if estabilizar.returncode == 0 else estabilizar.stderr
log += '\n'
assert estabilizar.returncode == 0, estabilizar.stderr
tablas, estados = [], []
try:
    for ronda, variantes in ORDEN.items():
        for posicion, variante in enumerate(variantes, 1):
            antes = subprocess.check_output(CONSULTA, text=True).strip()
            temporal = Path('build/fase6') / ronda / variante
            shutil.rmtree(temporal, ignore_errors=True)
            run = subprocess.run([PROGRAMA[variante], 'data/runway.pgm', '--runs', str(LANZAMIENTOS),
                                  '--output', str(temporal)], text=True, capture_output=True)
            despues = subprocess.check_output(CONSULTA, text=True).strip()
            estados += [f'{ronda},{variante},antes,{antes}', f'{ronda},{variante},despues,{despues}']
            resumen_programa = '\n'.join(l for l in run.stdout.splitlines() if not l.startswith('Ejecucion'))
            salida = f'== {ronda} {variante} (posicion {posicion}) | GPU antes: {antes} | despues: {despues}\n{resumen_programa}\n{run.stderr}'
            print(salida)
            log += salida
            assert run.returncode == 0, f'Fallo {variante} en {ronda}'
            completa = pd.read_csv(temporal / ARCHIVO[variante])
            assert len(completa) == LANZAMIENTOS and completa['validated'].all()
            assert completa['run'].tolist() == list(range(1, LANZAMIENTOS + 1))
            tramos = {'1-10': completa['run'] <= 10, '11-90': completa['run'].between(11, 90),
                      '91-100': completa['run'] > LANZAMIENTOS - REPORTADOS}
            log += 'Promedios por tramo (ms): ' + '; '.join(
                f"{nombre}: {completa.loc[filtro, 'kernel_ms'].mean():.6f}" for nombre, filtro in tramos.items()) + '\n'
            lineas = (temporal / ARCHIVO[variante]).read_text().splitlines()
            destino = out / ronda / ARCHIVO[variante]
            destino.parent.mkdir(exist_ok=True)
            destino.write_text('\n'.join([lineas[0], *lineas[-REPORTADOS:]]) + '\n')
            reportada = pd.read_csv(destino)
            assert reportada['run'].tolist() == list(range(LANZAMIENTOS - REPORTADOS + 1, LANZAMIENTOS + 1))
            tablas.append(reportada.assign(ronda=ronda, variante=variante, posicion=posicion))
finally:
    (out / 'fase6-validacion.txt').write_text(log)
    (out / 'estado-gpu.csv').write_text('ronda,variante,momento,pstate,sm_mhz,mem_mhz,temp_c,power_w\n'
                                       + '\n'.join(estados) + '\n')
tiempos = pd.concat(tablas, ignore_index=True)
display(tiempos.pivot_table(index='run', columns=['ronda', 'variante'], values='kernel_ms'))

## Recursos por kernel

`cuobjdump -res-usage` confirma la memoria constante y la memoria compartida estática que reserva cada kernel (se espera 720 bytes constantes y 36 000 bytes compartidos en la variante final). Sirve también para el análisis de ocupación.

In [ ]:
uso = ''
for variante, programa in PROGRAMA.items():
    r = subprocess.run(['cuobjdump', '-res-usage', programa], text=True, capture_output=True)
    uso += f'== {variante} ({programa})\n{r.stdout}{r.stderr}\n'
(out / 'res-usage.txt').write_text(uso)
print(uso)

## Comparación

Desviación estándar muestral. `diferencia_pct` es `(variante − referencia) / referencia`, negativa si la variante es más rápida; `speedup` es tiempo de la referencia / tiempo de la variante. `p_welch` prueba si la diferencia de promedios es significativa y `p_bonferroni` lo corrige por las 3 comparaciones. La fila `total` combina las 60 mediciones por variante. La tabla por posición revela si el orden de ejecución sesga los tiempos.

In [ ]:
try:
    from scipy.stats import ttest_ind
except ImportError:
    ttest_ind = None
resumen = tiempos.groupby(['ronda', 'variante'])['kernel_ms'].agg(['count', 'mean', 'std', 'min', 'max'])
total = tiempos.groupby('variante')['kernel_ms'].agg(['count', 'mean', 'std', 'min', 'max']).reindex(NOMBRES)
resumen = pd.concat([resumen, pd.concat({'total': total})])
resumen.to_csv(out / 'resumen.csv')
display(resumen)
PARES = [('constante', 'global'), ('tres_memorias', 'global'), ('tres_memorias', 'constante')]
filas = []
for ronda in [*ORDEN, 'total']:
    datos = tiempos if ronda == 'total' else tiempos.query('ronda == @ronda')
    for a, b in PARES:
        x = datos.query('variante == @a')['kernel_ms']
        y = datos.query('variante == @b')['kernel_ms']
        fila = dict(ronda=ronda, variante=a, referencia=b, variante_ms=x.mean(), referencia_ms=y.mean(),
                    diferencia_pct=100 * (x.mean() - y.mean()) / y.mean(), speedup=y.mean() / x.mean())
        if ttest_ind is not None:
            p = ttest_ind(y, x, equal_var=False).pvalue
            fila['p_welch'] = p
            fila['p_bonferroni'] = min(1.0, p * len(PARES))
        filas.append(fila)
comparacion = pd.DataFrame(filas)
comparacion.to_csv(out / 'comparacion.csv', index=False)
display(comparacion)
posicion = tiempos.pivot_table(index='variante', columns='posicion', values='kernel_ms', aggfunc='mean').reindex(NOMBRES)
posicion.to_csv(out / 'promedio-por-posicion.csv')
display(posicion)
display(pd.read_csv(out / 'estado-gpu.csv'))

In [ ]:
import matplotlib.pyplot as plt
ETIQUETAS = {'global': 'Global', 'constante': 'Global +\nconstante',
             'tres_memorias': 'Global + constante\n+ compartida'}
fig, ax = plt.subplots(figsize=(7, 4.5))
datos = [tiempos.loc[tiempos['variante'] == v, 'kernel_ms'] for v in NOMBRES]
ax.boxplot(datos, showmeans=True)
ax.set_xticks(range(1, len(NOMBRES) + 1))
ax.set_xticklabels([ETIQUETAS[v] for v in NOMBRES])
ax.set_ylabel('Tiempo del kernel (ms)')
ax.set_title('Transformada de Hough, 800 x 600: 60 mediciones por variante')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(out / 'comparacion-tres-memorias.png', dpi=150)
plt.show()

## Resultados

In [ ]:
from google.colab import files
import zipfile
bundle = Path('build/fase6-tres-memorias.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(out.rglob('*')):
        if path.is_file():
            archive.write(path, path.as_posix())
files.download(str(bundle))